# 04 - Optimal Commute Routing with Multi-Candidate Stations (Round-Trip)

This notebook calculates optimal door-to-door commute times using OpenTripPlanner for transit routing between multiple candidate stations for each employee and the office, selecting the combination that minimizes total round-trip commute time.

**How it works:**
- For each employee, we have 5 candidate stations from the previous notebook
- For the office, we have 5 candidate stations from the previous notebook
- We evaluate all 25 combinations (5 employee stations × 5 office stations)
- For each combination, we calculate both morning and evening journeys:
  - Morning: walk_home_to_station + transit_time_morning + walk_station_to_office
  - Evening: walk_office_to_station + transit_time_evening + walk_station_to_home
- Round-trip total = morning journey + evening journey
- We select the combination with the minimal round-trip total commute time

**Formula:** (walk_home_to_station + transit_time_morning + walk_station_to_office) + (walk_office_to_station + transit_time_evening + walk_station_to_home)

In [1]:
# Import the libraries we need for routing
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os

In [2]:
# Load environment variables and configure OpenTripPlanner server
load_dotenv()

# OpenTripPlanner server configuration
OTP_BASE_URL = "http://localhost:8080/otp/routers/default"

# Load the office candidate stations with walking times from the previous notebook
office_candidates_df = pd.read_csv('data/jj_office_candidate_stations_with_walking.csv')
print(f"Loaded {len(office_candidates_df)} office candidate stations")
print(office_candidates_df[['station_name', 'rank', 'walking_time_min']])

# Load the employee candidate stations with walking times from the previous notebook
employee_candidates_df = pd.read_csv('data/employee_candidate_stations_with_walking.csv')
print(f"\nLoaded {len(employee_candidates_df)} employee candidate stations")
print(f"Employees with candidates: {employee_candidates_df['employee_id'].nunique()}")

# Display sample data to verify
print("\nSample employee candidates:")
print(employee_candidates_df[['employee_id', 'station_name', 'rank', 'walking_time_min']].head(10))

Loaded 5 office candidate stations
                     station_name  rank  walking_time_min
0       Harksheide, Libellengrund     1          3.706667
1       Harksheide, Libellengrund     2          3.716667
2       Harksheide, Libellengrund     3          3.720000
3  Harksheide, Heidehofweg (West)     4          6.415000
4  Harksheide, Heidehofweg (West)     5          6.403333

Loaded 880 employee candidate stations
Employees with candidates: 176

Sample employee candidates:
  employee_id                      station_name  rank  walking_time_min
0      EMP001                          Fährdamm     1         28.505000
1      EMP001                      Zimmerstraße     2          8.410000
2      EMP001                      Zimmerstraße     3          8.108333
3      EMP001                      Zimmerstraße     4          8.543333
4      EMP001                      Zimmerstraße     5          8.541667
5      EMP002  Friedrichsgabe, Langenharmer Weg     1          6.001667
6      EMP002

In [3]:
# Load the original employee data to store our final routing results
# Check if we have existing data with car routing info to preserve it
if os.path.exists('data/synthetic_employees_with_optimal_commutes.csv'):
    employee_df = pd.read_csv('data/synthetic_employees_with_optimal_commutes.csv')
    print(f"Loaded {len(employee_df)} employees from existing commute data (preserving car routing data)")
else:
    employee_df = pd.read_csv('data/synthetic_employees_geocoded.csv')
    print(f"Loaded {len(employee_df)} employees from geocoded data")

Loaded 178 employees from existing commute data (preserving car routing data)


In [4]:
# Function to get transit route between two stations using OpenTripPlanner API
def get_transit_route(from_lat, from_lon, to_lat, to_lon, date, time_str):
    """Get transit route using OpenTripPlanner API"""
    url = f"{OTP_BASE_URL}/plan"
    
    params = {
        'fromPlace': f"{from_lat},{from_lon}",
        'toPlace': f"{to_lat},{to_lon}",
        'mode': 'TRANSIT,WALK',
        'date': date,
        'time': time_str
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        if 'error' in data and data['error']['id'] == 406:
            # No transit times available - this is okay, return walking only
            return data
        
        return data
        
    except Exception as e:
        print(f"Error fetching route: {e}")
        return None

In [5]:
# Test the routing function with a sample employee candidate and office candidate
sample_employee_candidate = employee_candidates_df.iloc[0]
sample_office_candidate = office_candidates_df.iloc[0]

print(f"Testing routing with sample stations:")
print(f"Employee: {sample_employee_candidate['employee_id']}")
print(f"Employee station: {sample_employee_candidate['station_name']} (rank {sample_employee_candidate['rank']})")
print(f"Office station: {sample_office_candidate['station_name']} (rank {sample_office_candidate['rank']})")

from_lat = sample_employee_candidate['station_lat']
from_lon = sample_employee_candidate['station_lon']
to_lat = sample_office_candidate['station_lat']
to_lon = sample_office_candidate['station_lon']

print(f"From: ({from_lat}, {from_lon})")
print(f"To: ({to_lat}, {to_lon})")

# Set date for routing (within GTFS range - our GTFS data has April-December 2026)
route_date = "2026-07-08"
morning_time = "08:00"
evening_time = "17:00"

print(f"\n--- Morning Journey (Home to Office) ---")
route_data_morning = get_transit_route(from_lat, from_lon, to_lat, to_lon, route_date, morning_time)

if route_data_morning and 'plan' in route_data_morning:
    if route_data_morning['plan']['itineraries']:
        itinerary = route_data_morning['plan']['itineraries'][0]
        print(f"Transit time: {itinerary['transitTime']}s ({itinerary['transitTime']/60:.1f} min)")
        print(f"Walking time: {itinerary['walkTime']}s ({itinerary['walkTime']/60:.1f} min)")
        print(f"Total time: {itinerary['duration']}s ({itinerary['duration']/60:.1f} min)")
        
        # Calculate total commute using our walking data from the previous notebook
        employee_walk = sample_employee_candidate['walking_time_min']
        office_walk = sample_office_candidate['walking_time_min']
        morning_total = employee_walk + (itinerary['transitTime']/60) + office_walk
        print(f"Using our walking data: {employee_walk:.1f} + {itinerary['transitTime']/60:.1f} + {office_walk:.1f} = {morning_total:.1f} min")
        transit_time_morning = itinerary['transitTime'] / 60
    else:
        print("No itineraries found")
        transit_time_morning = None
else:
    print("Error in routing")
    transit_time_morning = None

print(f"\n--- Evening Journey (Office to Home) ---")
route_data_evening = get_transit_route(to_lat, to_lon, from_lat, from_lon, route_date, evening_time)

if route_data_evening and 'plan' in route_data_evening:
    if route_data_evening['plan']['itineraries']:
        itinerary = route_data_evening['plan']['itineraries'][0]
        print(f"Transit time: {itinerary['transitTime']}s ({itinerary['transitTime']/60:.1f} min)")
        print(f"Walking time: {itinerary['walkTime']}s ({itinerary['walkTime']/60:.1f} min)")
        print(f"Total time: {itinerary['duration']}s ({itinerary['duration']/60:.1f} min)")
        
        # Calculate total commute using our walking data from the previous notebook
        evening_total = office_walk + (itinerary['transitTime']/60) + employee_walk
        print(f"Using our walking data: {office_walk:.1f} + {itinerary['transitTime']/60:.1f} + {employee_walk:.1f} = {evening_total:.1f} min")
        transit_time_evening = itinerary['transitTime'] / 60
    else:
        print("No itineraries found")
        transit_time_evening = None
else:
    print("Error in routing")
    transit_time_evening = None

if transit_time_morning is not None and transit_time_evening is not None:
    round_trip_total = morning_total + evening_total
    print(f"\n--- Round-Trip Summary ---")
    print(f"Morning: {morning_total:.1f} min")
    print(f"Evening: {evening_total:.1f} min")
    print(f"Round-trip total: {round_trip_total:.1f} min")

Testing routing with sample stations:
Employee: EMP001
Employee station: Fährdamm (rank 1)
Office station: Harksheide, Libellengrund (rank 1)
From: (53.5736668, 10.002784)
To: (53.687363, 10.0089284)

--- Morning Journey (Home to Office) ---
Transit time: 1440s (24.0 min)
Walking time: 2534s (42.2 min)
Total time: 3974s (66.2 min)
Using our walking data: 28.5 + 24.0 + 3.7 = 56.2 min

--- Evening Journey (Office to Home) ---
Transit time: 1440s (24.0 min)
Walking time: 2527s (42.1 min)
Total time: 3967s (66.1 min)
Using our walking data: 3.7 + 24.0 + 28.5 = 56.2 min

--- Round-Trip Summary ---
Morning: 56.2 min
Evening: 56.2 min
Round-trip total: 112.4 min


In [6]:
# Set up the multi-candidate routing process
print("Starting multi-candidate optimal routing (round-trip)...")
print(f"Employees: {employee_df['employee_id'].nunique()}")
print(f"Employee candidates per employee: 5")
print(f"Office candidates: {len(office_candidates_df)}")
print(f"Total combinations to evaluate: {employee_df['employee_id'].nunique() * 5 * 5}")

print(f"\nUsing date: {route_date}")
print(f"Morning time: {morning_time}, Evening time: {evening_time}")
print(f"Formula: (walk_home_to_station + transit_morning + walk_station_to_office) + (walk_office_to_station + transit_evening + walk_station_to_home)")

Starting multi-candidate optimal routing (round-trip)...
Employees: 178
Employee candidates per employee: 5
Office candidates: 5
Total combinations to evaluate: 4450

Using date: 2026-07-08
Morning time: 08:00, Evening time: 17:00
Formula: (walk_home_to_station + transit_morning + walk_station_to_office) + (walk_office_to_station + transit_evening + walk_station_to_home)


In [7]:
# Main routing loop - evaluate all station combinations for each employee (round-trip)


employees_processed = 0
combinations_evaluated = 0
cache_file = 'data/routing_cache.csv'

# Load cache if exists to avoid recalculating the same routes
if os.path.exists(cache_file):
    print("Loading routing cache...")
    routing_cache = pd.read_csv(cache_file)
    print(f"Loaded {len(routing_cache)} cached routes")
else:
    routing_cache = pd.DataFrame(columns=['cache_key', 'employee_station_id', 'office_station_id', 'transit_time_morning_min', 'transit_time_evening_min', 'route_found'])
    print("No cache found, starting fresh")

print("\nProcessing employees...")

# Process each employee individually
for employee_id in employee_df['employee_id']:
    print(f"\nProcessing {employee_id} ({employees_processed + 1}/{len(employee_df)})...")
    
    # Get this employee's 5 candidate stations
    emp_candidates = employee_candidates_df[employee_candidates_df['employee_id'] == employee_id]
    
    best_total_time = float('inf')
    best_route = None
    
    # Evaluate all combinations of employee station × office station (5 × 5 = 25 combinations)
    for _, emp_candidate in emp_candidates.iterrows():
        for _, office_candidate in office_candidates_df.iterrows():
            
            # Check if this combination is already cached
            cache_key = f"{emp_candidate['station_id']}_{office_candidate['station_id']}"
            if len(routing_cache) > 0 and cache_key in routing_cache['cache_key'].values:
                # Use cached transit time to save API calls
                cached_route = routing_cache[routing_cache['cache_key'] == cache_key].iloc[0]
                transit_time_morning = cached_route['transit_time_morning_min']
                transit_time_evening = cached_route['transit_time_evening_min']
                route_found = cached_route['route_found']
            else:
                # Calculate morning transit time via OpenTripPlanner (home to office)
                route_data_morning = get_transit_route(
                    emp_candidate['station_lat'], emp_candidate['station_lon'],
                    office_candidate['station_lat'], office_candidate['station_lon'],
                    route_date, morning_time
                )
                
                # Calculate evening transit time via OpenTripPlanner (office to home)
                route_data_evening = get_transit_route(
                    office_candidate['station_lat'], office_candidate['station_lon'],
                    emp_candidate['station_lat'], emp_candidate['station_lon'],
                    route_date, evening_time
                )
                
                # Extract transit times
                if route_data_morning and 'plan' in route_data_morning and route_data_morning['plan']['itineraries']:
                    itinerary_morning = route_data_morning['plan']['itineraries'][0]
                    transit_time_morning = itinerary_morning['transitTime'] / 60
                else:
                    transit_time_morning = None
                
                if route_data_evening and 'plan' in route_data_evening and route_data_evening['plan']['itineraries']:
                    itinerary_evening = route_data_evening['plan']['itineraries'][0]
                    transit_time_evening = itinerary_evening['transitTime'] / 60
                else:
                    transit_time_evening = None
                
                # Route is found only if both directions have valid transit times
                route_found = (transit_time_morning is not None) and (transit_time_evening is not None)
                
                # Add to cache
                new_cache_entry = pd.DataFrame([{
                    'cache_key': cache_key,
                    'employee_station_id': emp_candidate['station_id'],
                    'office_station_id': office_candidate['station_id'],
                    'transit_time_morning_min': transit_time_morning,
                    'transit_time_evening_min': transit_time_evening,
                    'route_found': route_found
                }])
                routing_cache = pd.concat([routing_cache, new_cache_entry], ignore_index=True)
            
            # Calculate round-trip commute time for this combination
            if route_found and transit_time_morning is not None and transit_time_evening is not None:
                # Morning journey: walk_home_to_station + transit_morning + walk_station_to_office
                morning_journey = emp_candidate['walking_time_min'] + transit_time_morning + office_candidate['walking_time_min']
                
                # Evening journey: walk_office_to_station + transit_evening + walk_station_to_home
                evening_journey = office_candidate['walking_time_min'] + transit_time_evening + emp_candidate['walking_time_min']
                
                # Round-trip total
                round_trip_time = morning_journey + evening_journey
                
                # Track the best route for this employee
                if round_trip_time < best_total_time:
                    best_total_time = round_trip_time
                    best_route = {
                        'employee_station': emp_candidate['station_name'],
                        'employee_station_id': emp_candidate['station_id'],
                        'employee_station_rank': emp_candidate['rank'],
                        'office_station': office_candidate['station_name'],
                        'office_station_id': office_candidate['station_id'],
                        'office_station_rank': office_candidate['rank'],
                        'walk_home_to_station_min': emp_candidate['walking_time_min'],
                        'transit_time_morning_min': transit_time_morning,
                        'walk_station_to_office_min': office_candidate['walking_time_min'],
                        'walk_office_to_station_min': office_candidate['walking_time_min'],
                        'transit_time_evening_min': transit_time_evening,
                        'walk_station_to_home_min': emp_candidate['walking_time_min'],
                        'morning_journey_min': morning_journey,
                        'evening_journey_min': evening_journey,
                        'total_commute_min': round_trip_time,
                        'otp_route_found': True
                    }
            
            combinations_evaluated += 1
    
    # Store the best route for this employee (preserve existing car routing data)
    if best_route:
        employee_idx = employee_df[employee_df['employee_id'] == employee_id].index[0]
        for key, value in best_route.items():
            employee_df.at[employee_idx, key] = value
    else:
        # No valid route found for this employee
        employee_idx = employee_df[employee_df['employee_id'] == employee_id].index[0]
        employee_df.at[employee_idx, 'otp_route_found'] = False
    
    employees_processed += 1
    
    # Save cache every 10 employees to avoid losing progress
    if employees_processed % 10 == 0:
        routing_cache.to_csv(cache_file, index=False)
        print(f"\nProgress: {employees_processed}/{len(employee_df)} employees processed")
        print(f"Combinations evaluated: {combinations_evaluated}")
        print(f"Cache size: {len(routing_cache)} routes")

# Save final cache
routing_cache.to_csv(cache_file, index=False)
print(f"\n✅ Routing complete! Processed {employees_processed} employees, evaluated {combinations_evaluated} combinations.")

Loading routing cache...
Loaded 1325 cached routes

Processing employees...

Processing EMP001 (1/178)...

Processing EMP002 (2/178)...

Processing EMP003 (3/178)...

Processing EMP004 (4/178)...

Processing EMP005 (5/178)...

Processing EMP006 (6/178)...

Processing EMP007 (7/178)...

Processing EMP008 (8/178)...

Processing EMP009 (9/178)...

Processing EMP010 (10/178)...

Progress: 10/178 employees processed
Combinations evaluated: 250
Cache size: 1325 routes

Processing EMP011 (11/178)...

Processing EMP012 (12/178)...

Processing EMP013 (13/178)...

Processing EMP014 (14/178)...

Processing EMP015 (15/178)...

Processing EMP016 (16/178)...

Processing EMP017 (17/178)...

Processing EMP018 (18/178)...

Processing EMP019 (19/178)...

Processing EMP020 (20/178)...

Progress: 20/178 employees processed
Combinations evaluated: 500
Cache size: 1325 routes

Processing EMP021 (21/178)...

Processing EMP022 (22/178)...

Processing EMP023 (23/178)...

Processing EMP024 (24/178)...

Processi

In [8]:
# Summary of routing results
print(f"\n=== Routing Summary (Round-Trip) ===")
print(f"Total employees processed: {employees_processed}")
print(f"Total combinations evaluated: {combinations_evaluated}")
print(f"Employees with valid routes: {employee_df['otp_route_found'].sum()}")
print(f"Employees without trips: {(~employee_df['otp_route_found']).sum()}")

print(f"\nAverage round-trip commute time for employees with routes: {employee_df[employee_df['otp_route_found']]['total_commute_min'].mean():.1f} minutes")
print(f"Average morning journey: {employee_df[employee_df['otp_route_found']]['morning_journey_min'].mean():.1f} minutes")
print(f"Average evening journey: {employee_df[employee_df['otp_route_found']]['evening_journey_min'].mean():.1f} minutes")

# Display sample of optimal routes
print(f"\nSample optimal routes:")
print(employee_df[['employee_id', 'employee_station', 'employee_station_rank', 
                   'office_station', 'office_station_rank', 
                   'morning_journey_min', 'evening_journey_min', 'total_commute_min']].head(10))


=== Routing Summary (Round-Trip) ===
Total employees processed: 178
Total combinations evaluated: 4400
Employees with valid routes: 131
Employees without trips: 47

Average round-trip commute time for employees with routes: 118.4 minutes
Average morning journey: 59.7 minutes
Average evening journey: 58.7 minutes

Sample optimal routes:
  employee_id                 employee_station  employee_station_rank  \
0      EMP001                     Zimmerstraße                    3.0   
1      EMP002                              NaN                    NaN   
2      EMP003                              NaN                    NaN   
3      EMP004                Rübke, Kurzer Weg                    1.0   
4      EMP005                     Zimmerstraße                    3.0   
5      EMP006  Eißendorfer Straße (TU Hamburg)                    5.0   
6      EMP007               Alte Holstenstraße                    1.0   
7      EMP008             Ehestorf, Hohlredder                    3.0   
8   

In [9]:
# Save the final results with optimal commute routes
employee_df.to_csv('data/synthetic_employees_with_optimal_commutes.csv', index=False)
print("Saved optimal commute results to data/synthetic_employees_with_optimal_commutes.csv")

Saved optimal commute results to data/synthetic_employees_with_optimal_commutes.csv
